# **MODEL 1: Multimodal Integration - Late Fusion (EfficientNetB1 + Metadata MLP)**

**Objective:**
Inclusion of **clinical metadata**
- **Age**
- **Sex**
- **Atomic location** of the lesion
<br>

**Methodology:**
| Variable | Strategy | NA Treatment |
|----------|-----------|--------------------|
| Age | Normalization (min-max) | Imputation by median |
| Sex | One-hot encoding | Category "unknown" |
| Location | One-hot encoding (15 sítios) | Category "unknown" |

### Imports

In [2]:
import os, sys, json
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

### **Data** Configuration

In [3]:
if os.getcwd().endswith('models'):
    os.chdir('..')

In [4]:
BASE_PATH = "./data"
TRAIN_CSV = os.path.join(BASE_PATH, "augmented_metadata.csv")
VAL_CSV = os.path.join(BASE_PATH, "val_split.csv")
TEST_CSV = os.path.join(BASE_PATH, "test_split.csv")

In [5]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

### **Data** Loading & Mapping

In [9]:
def preprocess_metadata(df, scaler=None, encoder=None, is_training=True):
    # Impute missing values
    df['age'] = df['age'].fillna(df['age'].median())
    df['sex'] = df['sex'].fillna('unknown')
    df['localization'] = df['localization'].fillna('unknown')
    
    # Scale Age
    if is_training:
        scaler = MinMaxScaler()
        age_scaled = scaler.fit_transform(df[['age']])
    else:
        age_scaled = scaler.transform(df[['age']])
        
    # One-Hot Encode (Fix: sparse_output instead of sparse)
    cat_cols = ['sex', 'localization']
    if is_training:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        encoded_cats = encoder.fit_transform(df[cat_cols])
    else:
        encoded_cats = encoder.transform(df[cat_cols])
    
    # Create Meta Vector for the MLP branch
    meta_vectors = np.hstack([age_scaled, encoded_cats])
    return meta_vectors, scaler, encoder

## Metadata preprocessing

Clean and encode the clinical features (Age, Sex, Localization) into a fixed-length vector before the split.

In [10]:
train_meta, scaler, encoder = preprocess_metadata(train_df, is_training=True)
val_meta, _, _ = preprocess_metadata(val_df, scaler, encoder, is_training=False)
test_meta, _, _ = preprocess_metadata(test_df, scaler, encoder, is_training=False)

## 4. Multimodal Data Pipeline

In [11]:
# This ensures Stream A (Image) and Stream B (Meta) stay connected
def load_multimodal_item(path, meta, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [240, 240]) # EfficientNetB1
    img = tf.cast(img, tf.float32) / 255.0
    return {"image_input": img, "meta_input": meta}, label

In [12]:
def create_ds(df, meta):
    return tf.data.Dataset.from_tensor_slices((
        df['image_path'].values, 
        meta, 
        df['dx_encoded'].values.astype(np.int32)
    )).map(load_multimodal_item).batch(32).prefetch(tf.data.AUTOTUNE)

In [13]:
train_ds = create_ds(train_df, train_meta)
val_ds = create_ds(val_df, val_meta)
test_ds = create_ds(test_df, test_meta)

## 5. Architecture Late Fusion: EfficientNetB1 + Metadata MLP 

This builds the two branches and concatenates them into the final classification head. Establishing correct connections between EfficientNet and the MLP

In [14]:
# Image Stream (EfficientNetB1)
base_model = tf.keras.applications.EfficientNetB1(include_top=False, weights='imagenet', input_shape=(240, 240, 3))
image_input = layers.Input(shape=(240, 240, 3), name="image_input")
x = base_model(image_input)
x = layers.GlobalAveragePooling2D()(x)
image_features = layers.Dense(256, activation='relu')(x)

27018416/27018416 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step


In [16]:
# Metadata Stream (MLP)
meta_input = layers.Input(shape=(train_meta.shape[1],), name="meta_input")
y = layers.Dense(64, activation='relu')(meta_input)
meta_features = layers.Dense(32, activation='relu')(y)

In [17]:
# Concatenation (The Fusion Point)
combined = layers.Concatenate()([image_features, meta_features])
final_output = layers.Dense(7, activation='softmax')(combined)

model = models.Model(inputs=[image_input, meta_input], outputs=final_output)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

Training

In [18]:
model.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10


KeyboardInterrupt: 